# PHM 2015 | Preparing Measurements for Active-Event Diagnosis

A case asks which target event was logged at a site at decision time `t`. Its evidence consists of component readings available in the preceding three hours. This notebook builds the measurement and private scoring tables, freezes the site assignments, and reproduces a worked case before preparing a small training sample.

Run the cells in order with the project Python kernel. The full pass reads all 210 source file entries and processes the component and event streams. Zone availability is reported separately. The [project guide](../CP5403_A2_Project_Guide.md) defines the task. The [version history](version_history/v0.1.0_preprocessing.md) explains each table and how to reuse it.

## 1. Locate the Data and Set the Memory Budget

The raw directory and the `Files` project share a parent. DuckDB uses one worker, a 256 MB database memory budget, and temporary disk space for sorts. Python holds a site clock and one component's coverage vector at a time. The database budget applies to DuckDB's managed buffers. The kernel and its Python objects also use memory.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import sys

import duckdb
import pandas as pd
from IPython.display import display

project = Path.cwd().resolve()
if (project / "Files" / "phm_preprocessing.py").is_file():
    project = project / "Files"
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import phm_preprocessing as prep

raw = project.parent / "Datasets" / "PHM15-master"
assert raw.is_dir(), f"Missing source directory: {raw}"
output = project.parent / "Datasets" / "PHM15-prepared"
output.mkdir(exist_ok=True)
destination = output / "case_inputs.duckdb"
scratch = TemporaryDirectory(prefix="build-", dir=output)
staged = Path(scratch.name) / destination.name
db = duckdb.connect(str(staged), config={"threads": 1, "memory_limit": "256MB"})
started = perf_counter()
print(f"Kernel {sys.executable}")
print(f"Python {sys.version.split()[0]} | DuckDB {duckdb.__version__} | pandas {pd.__version__}")

Kernel C:\Users\ges2j\OneDrive\Desktop\Assessments\CP5403\Files\.venv\Scripts\python.exe
Python 3.14.8 | DuckDB 2.0.0 | pandas 3.1.0


## 2. Audit the Sites and Align the Component Clock

Files ending in `a`, `b`, and `c` contain component readings, zone readings, and event logs, respectively. For a bin ending at 10:15, a reading at 10:00 belongs to the previous bin and one at 10:15 belongs to this bin. Logging jitter can put two readings into one bin. The later actual timestamp wins, with source row as the tie-breaker. All channels and the operating state come from that single row.

The parser keeps finite numeric values in their original scale. Missing channels remain null. A missing zone file leaves the component stream eligible. The audit distinguishes a missing file, an empty file, and a populated file whose records cannot be used.

In [2]:
prep.prepare_schema(db)
prep.inventory_sites(db, raw)

display(db.execute('''select component_file, zone_file, event_file, count(*) as sites
    from site_inventory group by all order by all''').df())
display(db.execute('''select site_id, component_file, zone_file, event_file, exclusion
    from site_inventory where exclusion is not null order by site_id''').df())
display(db.execute('''select sum(component_rows)::bigint as raw_readings,
    sum(malformed_rows)::bigint as malformed, sum(bad_keys)::bigint as invalid_keys, sum(bad_numeric_cells)::bigint as invalid_numbers,
    sum(superseded_rows)::bigint as superseded, sum(retained_bins)::bigint as retained
    from site_inventory''').df())
display(db.execute('select * from component_rejects order by site_id, file_line limit 10').df())

Audited 10 of 70 sites


Audited 20 of 70 sites


Audited 30 of 70 sites


Audited 40 of 70 sites


Audited 50 of 70 sites


Audited 60 of 70 sites


Audited 70 of 70 sites


,component_file,zone_file,event_file,sites
0,empty,empty,empty,1
1,empty,empty,present,3
2,present,empty,present,8
3,present,present,present,58


,site_id,component_file,zone_file,event_file,exclusion
0,50,empty,empty,present,no component bins
1,52,empty,empty,present,no component bins
2,62,empty,empty,empty,"no component bins, no valid log intervals"
3,64,empty,empty,present,no component bins


,raw_readings,malformed,invalid_keys,invalid_numbers,superseded,retained
0,32655753,1,0,0,1122963,31532789


,site_id,file_line,raw_text,reason
0,2,635661,"HVAC10,2",Expected Number of Columns: 10 Found: 2


## 3. Resolve the Target Intervals

A target is active from its start up to its end, excluding the end. Duplicate and overlapping intervals of the same target form one episode. Touching intervals remain separate because their half-open time ranges do not overlap. A missing target end censors subsequent decisions at that site through the end of its measurement record. The local release supplies no verified clearance markers.

The first table counts source intervals before merging. The second reports every event row by quality. The third counts target episodes too short or poorly aligned to intersect a quarter-hour tick, regardless of measurement coverage.

In [3]:
prep.merge_episodes(db)
display(db.execute('''select event_name, count(*) as raw_intervals, count(distinct site_id) as sites
    from private_events join site_inventory using (site_id)
    where retained_bins > 0 and quality='valid' and event_name in (?, ?)
      and end_at-start_at >= interval '15 minutes' group by event_name order by event_name''',
    list(prep.TARGETS)).df())
display(db.execute("select quality, count(*) as log_rows from private_events group by 1 order by 1").df())
display(db.execute('''select event_name, count(*) as merged_episodes,
    countif(not hits_clock_grid) as misses_grid from private_episodes group by 1 order by 1''').df())

,event_name,raw_intervals,sites
0,RuntimeExcessive,17458,65
1,SupplyTempTooCold,13811,53


,quality,log_rows
0,missing_end,49
1,nonpositive_duration,87
2,valid,178964


,event_name,merged_episodes,misses_grid
0,RuntimeExcessive,7180,6.0
1,SupplyTempTooCold,13386,127.0


## 4. Determine Eligibility and Freeze the Site Split

The clock runs from the first quarter hour on or after the first reading to the last quarter hour on or before the last reading. Each decision needs a full three-hour lookback and at least one component with eight populated first-channel bins out of twelve. This is a pilot coverage rule. The saved bins also support other channel-specific thresholds.

Both targets active, or an unresolved target, makes the label ambiguous. A control additionally needs a valid log start strictly before and another strictly after the decision, each within seven days. Logged `NOCOMMS` periods exclude controls. The release does not map original challenge test plants to known incomplete periods. The comparison class therefore means `NoRecordedTarget`. Review log completeness before freezing the evaluation controls.

In [4]:
sites = db.execute('''select site_id, first_reading, last_reading from site_inventory
    where exclusion is null order by site_id''').df()
for site in sites.itertuples():
    prep.prepare_decisions(db, site.site_id, site.first_reading, site.last_reading)

prep.freeze_split(db)
display(db.execute('''select split, count(*) as sites, min(seed) as seed
    from site_split group by split order by split''').df())
display(db.execute('''select status, count(*) as decision_ticks from private_decisions
    group by status order by decision_ticks desc''').df())
display(db.execute('''select disposition, count(*) as episodes from private_episode_audit
    group by disposition order by episodes desc''').df())

,split,sites,seed
0,test,10,5403
1,train,46,5403
2,validation,10,5403


,status,decision_ticks
0,eligible,3603774
1,control_unbracketed,667561
2,control_log_outage,90347
3,both_targets_active,12973
4,insufficient_ch1,6585
5,partial_history,203
6,unresolved_target,123


,disposition,episodes
0,available,18865
1,no_eligible_tick,1547
2,misses_grid,133
3,outside_measurements,21


## 5. Build Paired Training Examples

Each query summarises the twelve trailing bins. For each channel, compute each qualifying component's median, interquartile range, and last minus first value. Then report the median, lowest, and highest of those component statistics. Population counts and valid-bin counts preserve the coverage behind each number.

The development sample contains three examples per class, including the worked example. Each positive uses a distinct merged episode and the windows from a given site do not overlap. Only the private scoring record carries the absolute decision time, site identity and reference outcome.

In [5]:
import phm_case_samples as samples

samples.build_sample(db)
display(db.execute('''select reference_label, count(*) as sample_cases, count(distinct site_id) as sites
    from private_sample_scores group by 1 order by 1''').df())
display(db.execute('''select case_id, observed_components, usable_components_ch1,
    observed_component_bins_ch1, ch1_median_of_component_medians
    from sample_queries order by case_id limit 3''').df())

,reference_label,sample_cases,sites
0,NoRecordedTarget,3,3
1,RuntimeExcessive,3,3
2,SupplyTempTooCold,3,3


,case_id,observed_components,usable_components_ch1,observed_component_bins_ch1,ch1_median_of_component_medians
0,0062df7aabc85412ab2ea14d7f7bb94f,10,10,113,731.0
1,18012a7905115cb292230dd477980578,8,8,93,732.0
2,1d9043ef03d15c238b54f3ac921bd2d2,6,6,71,792.5


## 6. Reproduce the Site 1 Case From the Raw CSVs

At 18:30 on 7 June 2010, the window starts just after 15:30. An independent CSV pass assigns readings to bins by their age at the decision. Compare the resulting per-component counts and medians with the database calculation, then check which raw target interval contains 18:30.

In [6]:
component_check, event_check = samples.verify_worked_case(db, raw)
display(component_check)
display(event_check)

,component,raw_bins,raw_median,prepared_bins,prepared_median,usable
0,HVAC1,12,709.0,12,709.0,True
1,HVAC2,11,712.0,11,712.0,True
2,HVAC3,11,736.0,11,736.0,True
3,HVAC4,12,722.5,12,722.5,True
4,HVAC5,7,714.0,7,714.0,False
5,HVAC6,12,724.0,12,724.0,True


,raw_start,raw_end,recorded_target
0,2010-06-07 17:59:37,2010-06-07 18:59:58,RuntimeExcessive


## 7. Validate and Publish the Case Inputs

The final checks exercise the saved tables, including unique bin keys, half-open time boundaries, decision eligibility, complete query-to-score pairing, and site isolation in the training index. The verified database replaces the previous output in one file operation. The query table can be read on its own.

In [7]:
checks = samples.validate(db)
display(pd.DataFrame({"passed_check": checks}))
db.close()
staged.replace(destination)
scratch.cleanup()
print(f"Prepared {destination}")
print(f"{destination.stat().st_size / 2**20:,.1f} MiB on disk | {(perf_counter()-started)/60:.1f} minutes")

,passed_check
0,All 70 sites and three stream inventories reco...
1,Source rows and bin counts reconcile
2,Every retained reading lies in its right-close...
3,Component and bin keys are unique within every...
4,Merged same-target episodes have positive dura...
5,Every valid target source interval contributes...
6,Eligible positives match an active episode and...
7,Site assignments are disjoint with Site 1 in t...
8,All three classes have eligible ticks in each ...
9,Nine measurement queries pair one-to-one with ...


Prepared C:\Users\ges2j\OneDrive\Desktop\Assessments\CP5403\Datasets\PHM15-prepared\case_inputs.duckdb
471.3 MiB on disk | 5.7 minutes


## 8. Read a Case for the Next Stage

Use `component_bins` to compute new features, `private_decisions` to select eligible scoring points, and `private_episode_audit` to count available or lost positive episodes. Select historical neighbours from training sites only. Fit numeric imputation, scaling and reference ranges after choosing the training cases.

The example below opens the published file read-only and projects the guide's first-channel metrics. Join to the private score by opaque case ID only when scoring or auditing.

In [8]:
with duckdb.connect(str(destination), read_only=True) as result:
    display(result.execute('''select q.case_id, q.observed_components, q.usable_components_ch1,
        q.observed_component_bins_ch1, q.possible_component_bins_ch1,
        q.ch1_median_of_component_medians, q.ch1_lowest_component_median,
        q.ch1_highest_component_median
        from sample_queries q join private_sample_scores s using (case_id)
        where s.site_id=1 and s.decision_time=?''', [prep.DEMO_TIME]).df())

,case_id,observed_components,usable_components_ch1,observed_component_bins_ch1,possible_component_bins_ch1,ch1_median_of_component_medians,ch1_lowest_component_median,ch1_highest_component_median
0,fec61d36892150ee95371a72c2827916,6,5,65,72,722.5,709.0,736.0


### Training Channel Ranges

Inspect missing values and native-scale extremes on training sites before choosing any outlier treatment. A missing quarter hour is absent from the bin table. Null counts below describe only values in retained rows. The audit computes all seven ranges in a single aggregate scan.

In [9]:
fields = []
for channel in prep.CHANNELS:
    fields.extend([f"count(*) - count({channel})", f"min({channel})", f"max({channel})"])
with duckdb.connect(str(destination), read_only=True, config={"threads": 1, "memory_limit": "256MB"}) as result:
    values = result.execute(f'''select {', '.join(fields)}
        from component_bins join site_split using (site_id) where split='train' ''').fetchone()

profile = [(channel, *values[i * 3:i * 3 + 3]) for i, channel in enumerate(prep.CHANNELS)]
display(pd.DataFrame(profile, columns=["channel", "null_values", "minimum", "maximum"]))

,channel,null_values,minimum,maximum
0,ch1,0,-777.0,29761.0
1,ch2,0,-4595.0,6717.0
2,ch3,0,0.0,117.0
3,ch4,0,0.0,1023.0
4,ch5,0,-118.0,1839.0
5,ch6,0,20.0,149.0
6,ch7,0,9.0,149.0
